In [ ]:
!pip install pycaret

In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

RANDOM_STATE = 42
df = pd.read_csv("mushroom_cleaned.csv")
target = "class"
X = df.drop(columns=target)
y = df[target]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print(f"Train shape: {X_train.shape}; test shape: {X_test.shape}")

## Majority-class baseline

Every candidate model should beat this benchmark.

In [ ]:
majority_class = y_train.mode().iloc[0]
majority_predictions = np.full(len(y_test), majority_class)
baseline_results = pd.DataFrame([{
    "model": "Majority class",
    "accuracy": accuracy_score(y_test, majority_predictions),
    "balanced_accuracy": balanced_accuracy_score(y_test, majority_predictions),
    "f1_poisonous": f1_score(y_test, majority_predictions, pos_label="p", zero_division=0),
}])
baseline_results

## Logistic-regression baseline

All imputation and one-hot encoding are fitted on the training data inside the pipeline.

In [ ]:
numeric_features = X_train.select_dtypes(include="number").columns.tolist()
categorical_features = X_train.select_dtypes(exclude="number").columns.tolist()
preprocessor = ColumnTransformer([
    ("numeric", SimpleImputer(strategy="median", add_indicator=True), numeric_features),
    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]), categorical_features),
])
logistic_baseline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE)),
])
logistic_baseline.fit(X_train, y_train)
logistic_predictions = logistic_baseline.predict(X_test)
poisonous_index = list(logistic_baseline.classes_).index("p")
logistic_probabilities = logistic_baseline.predict_proba(X_test)[:, poisonous_index]
sklearn_results = pd.DataFrame([{
    "model": "Logistic regression",
    "accuracy": accuracy_score(y_test, logistic_predictions),
    "balanced_accuracy": balanced_accuracy_score(y_test, logistic_predictions),
    "f1_poisonous": f1_score(y_test, logistic_predictions, pos_label="p"),
    "roc_auc": roc_auc_score((y_test == "p").astype(int), logistic_probabilities),
}])
display(sklearn_results)
print(classification_report(y_test, logistic_predictions, target_names=["edible", "poisonous"]))

## Scikit-learn model comparison

Scikit-learn compares several candidate models with stratified cross-validation on the training split. The holdout test set remains untouched until the selected model is evaluated.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
candidate_models = {
    "Logistic regression": LogisticRegression(
        max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE
    ),
    "Random forest": RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
}

cv_results = []
candidate_pipelines = {}
for model_name, model in candidate_models.items():
    candidate_pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model),
    ])
    scores = cross_validate(
        candidate_pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring={"accuracy": "accuracy", "balanced_accuracy": "balanced_accuracy", "f1_poisonous": "f1"},
        n_jobs=-1,
    )
    cv_results.append({
        "model": model_name,
        "accuracy": scores["test_accuracy"].mean(),
        "balanced_accuracy": scores["test_balanced_accuracy"].mean(),
        "f1_poisonous": scores["test_f1_poisonous"].mean(),
    })
    candidate_pipelines[model_name] = candidate_pipeline

comparison_table = pd.DataFrame(cv_results).sort_values(
    "balanced_accuracy", ascending=False
).reset_index(drop=True)
comparison_table

In [ ]:
selected_model_name = comparison_table.loc[0, "model"]
selected_model = candidate_pipelines[selected_model_name]
selected_model.fit(X_train, y_train)
holdout_labels = selected_model.predict(X_test)
selected_results = pd.DataFrame([{
    "model": f"Scikit-learn selected model ({selected_model_name})",
    "accuracy": accuracy_score(y_test, holdout_labels),
    "balanced_accuracy": balanced_accuracy_score(y_test, holdout_labels),
    "f1_poisonous": f1_score(y_test, holdout_labels, pos_label="p"),
}])
display(pd.concat([baseline_results, sklearn_results, selected_results], ignore_index=True))
print(classification_report(y_test, holdout_labels, target_names=["edible", "poisonous"]))

## Optional PyCaret comparison

This preserves the original automated model comparison. PyCaret may require a compatible Python environment, so run these cells only after installing it successfully.

In [ ]:
from pycaret.classification import add_metric, compare_models, finalize_model, predict_model, pull, setup

from sklearn.metrics import balanced_accuracy_score

pycaret_train = X_train.copy()
pycaret_train[target] = y_train
pycaret_session = setup(
    data=pycaret_train,
    target=target,
    session_id=RANDOM_STATE,
    fold=5,
    numeric_imputation="median",
    categorical_imputation="most_frequent",
    verbose=False,
    html=False,
)
add_metric(
    id="balanced_accuracy",
    name="Balanced Accuracy",
    score_func=balanced_accuracy_score,
    target="pred",
    greater_is_better=True,
)
best_model = compare_models(sort="Balanced Accuracy", n_select=1)
pycaret_comparison_table = pull()
pycaret_comparison_table.head(10)

In [ ]:
final_pycaret_model = finalize_model(best_model)
holdout_predictions = predict_model(final_pycaret_model, data=X_test.copy())
pycaret_labels = holdout_predictions["prediction_label"]
pycaret_results = pd.DataFrame([{
    "model": "PyCaret selected model",
    "accuracy": accuracy_score(y_test, pycaret_labels),
    "balanced_accuracy": balanced_accuracy_score(y_test, pycaret_labels),
    "f1_poisonous": f1_score(y_test, pycaret_labels, pos_label="p"),
}])
pd.concat([baseline_results, sklearn_results, selected_results, pycaret_results], ignore_index=True)

Use the holdout metrics for the final comparison. Because poisonous mushrooms are the safety-critical class, inspect poisonous recall and the confusion matrix before selecting a production model.